# 生成式AI技術與應用

## 學習目標

完成本 Notebook 後，你將能夠：

1. 說明生成式 AI 與鑑別式 AI 的差異。
2. 使用輕量文字特徵方法模擬生成式 AI 的「語意關聯」概念。
3. 以簡單的 n-gram 方法示範文字生成的基本想法。
4. 理解模型部署後可能面臨的資料漂移與版本回退風險。
5. 建立一個簡化版的 AI 服務監控與容錯流程。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節會用到的 Colab 預裝套件，並建立可重現的隨機種子。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from collections import defaultdict, Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)

print('環境設定完成')
print('可使用套件：numpy、pandas、matplotlib、sklearn、re、collections')


## 核心概念說明

生成式 AI 的重點是「根據輸入產生新內容」，例如文字、圖像、語音、影片或程式碼。相對地，鑑別式 AI 的重點是「判斷輸入屬於哪一類」，例如垃圾郵件分類、影像辨識或疾病預測。

在真實的大型語言模型中，系統會使用大量資料與深度神經網路學習語意結構。本 Notebook 不使用大型模型，而是用 TF-IDF、餘弦相似度與 n-gram 這些輕量方法，模擬生成式 AI 背後幾個重要概念：

- 文字可以被轉成數值特徵。
- 相似的輸入可以找到相近的知識片段。
- 模型可以根據既有語料產生新的句子。
- 部署後需要監控輸入資料是否改變。
- 異常時需要有版本回退或預設邏輯。


In [ ]:
# ── 示範：生成式 AI 與鑑別式 AI 的任務差異 ─────────────────
# 這段程式碼用表格整理生成式 AI 與鑑別式 AI 的差異，協助建立考試常見比較題的判斷基礎。

import pandas as pd

data = [
    ['目標', '產生新的內容', '判斷、分類或預測'],
    ['輸入輸出', '輸入提示詞，輸出文章、圖片描述或程式碼', '輸入資料，輸出類別或分數'],
    ['學習重點', '學習資料分佈與內容結構', '學習類別邊界或預測規則'],
    ['常見模型', 'GPT、GAN、VAE、Diffusion Models', 'SVM、決策樹、Logistic Regression'],
    ['應用例子', '摘要生成、文案撰寫、圖片生成', '影像分類、詐欺偵測、客戶流失預測']
]

df = pd.DataFrame(data, columns=['比較面向', '生成式 AI', '鑑別式 AI'])
print(df.to_string(index=False))


## 用 TF-IDF 模擬語意檢索

大型語言模型通常會把文字轉成向量表示，讓模型可以比較語意相近程度。本練習使用較輕量的 TF-IDF 取代真正的 Embedding。

TF-IDF 會依照「詞彙在單一文件出現的頻率」與「該詞彙在所有文件中的稀有程度」計算權重。中文沒有空白可以斷詞，因此下方程式碼把分析單位設成字元（`analyzer='char'`、`ngram_range=(2, 3)`），改用連續 2～3 個字的字元組合取代詞彙，再用餘弦相似度找出最接近使用者問題的知識片段。這不是完整的生成式 AI，但可以模擬許多 AI 應用中的「檢索增強生成」前置步驟。


In [ ]:
# ── 示範：用 TF-IDF 找出最相關知識片段 ───────────────────
# 這段程式碼示範如何將文字轉成向量，並找出與使用者提問最相近的教材內容。

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

knowledge = [
    '生成式 AI 可以根據提示詞產生文字、圖像、語音、影片或程式碼。',
    '鑑別式 AI 主要用於分類、辨識或預測，例如影像分類與疾病預測。',
    'Transformer 使用自注意力機制，能有效處理長距離語境關係。',
    '擴散模型常用於圖像生成，透過逐步去雜訊產生高品質影像。',
    '部署生成式 AI 系統時，需要監控資料漂移、延遲、成本與模型版本。'
]

query = '模型上線後為什麼要監控資料漂移和版本？'

vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 3))
X = vectorizer.fit_transform(knowledge + [query])

similarities = cosine_similarity(X[-1], X[:-1]).flatten()
best_idx = similarities.argmax()

result = pd.DataFrame({
    '知識片段': knowledge,
    '相似度': similarities.round(3)
}).sort_values('相似度', ascending=False)

print(result.to_string(index=False))
print('\n最相關片段：')
print(knowledge[best_idx])


## 從文字生成到部署風險

生成式 AI 應用不只要能產生內容，也要能穩定部署與維運。考試常見風險包括：

- 資源需求高：大型模型可能需要 GPU、TPU 或大量記憶體。
- 批次處理延遲：即時服務若延遲過高，會影響使用者體驗。
- 版本管理困難：不同模型版本可能產生不同答案。
- 資料漂移：輸入資料分佈改變後，模型表現可能下降。
- 回退機制不足：新版本異常時，服務需要能回到穩定版本。

以下先用簡單的 n-gram 模擬文字生成，理解生成式模型如何依既有語料產生新內容。


In [ ]:
# ── 示範：用 Bigram 模擬文字生成 ──────────────────────
# 這段程式碼用前一個詞預測下一個詞，示範生成式模型根據既有語料產生新句子的基本概念。

import re
import numpy as np
from collections import defaultdict, Counter

np.random.seed(7)

corpus = [
    '生成式 AI 可以 產生 文字 內容',
    '生成式 AI 可以 產生 圖像 內容',
    '大型 語言 模型 可以 回答 問題',
    '提示 工程 可以 改善 生成 品質',
    '模型 部署 需要 監控 延遲 與 成本',
    '資料 漂移 可能 造成 模型 表現 下降'
]

transitions = defaultdict(Counter)
for sentence in corpus:
    tokens = sentence.split()
    for current_word, next_word in zip(tokens[:-1], tokens[1:]):
        transitions[current_word][next_word] += 1

def generate_text(start_word, max_words=8):
    words = [start_word]
    current = start_word
    for _ in range(max_words - 1):
        if current not in transitions:
            break
        candidates = list(transitions[current].keys())
        counts = np.array(list(transitions[current].values()), dtype=float)
        probs = counts / counts.sum()
        current = np.random.choice(candidates, p=probs)
        words.append(current)
    return ' '.join(words)

for start in ['生成式', '模型', '資料']:
    print(start, '=>', generate_text(start))


## 資料漂移偵測與模型回退

生成內容之外，上線後的監控同樣是考點。以下模擬 AI 服務的監控流程：用 Jensen-Shannon 距離比較新資料與訓練資料的分佈差異，差異過大就回退到穩定的模型版本。


In [ ]:
# ── 實際應用：資料漂移偵測與模型回退 ────────────────────────
# 這段程式碼模擬 AI 服務上線後的監控流程：若新資料分佈與訓練資料差異過大，就回退到穩定模型版本。

import numpy as np
import pandas as pd
from scipy.spatial.distance import jensenshannon

np.random.seed(11)

# 模擬訓練時的輸入資料：大多是一般亮度影像
train_brightness = np.random.normal(loc=120, scale=20, size=1000)

# 模擬上線後的新資料：場域變暗，可能代表攝影機角度、光照或設備狀態改變
production_brightness = np.random.normal(loc=80, scale=25, size=1000)

bins = np.linspace(0, 255, 21)
train_hist, _ = np.histogram(train_brightness, bins=bins, density=True)
prod_hist, _ = np.histogram(production_brightness, bins=bins, density=True)

train_hist = train_hist + 1e-8
prod_hist = prod_hist + 1e-8

drift_score = jensenshannon(train_hist, prod_hist)
threshold = 0.18

active_model = 'model_v2_candidate'
stable_model = 'model_v1_stable'

if drift_score > threshold:
    decision = '偵測到資料漂移，啟動回退機制'
    active_model = stable_model
else:
    decision = '資料分佈穩定，維持目前模型'

summary = pd.DataFrame({
    '監控項目': ['資料漂移分數', '門檻值', '系統決策', '目前使用模型'],
    '結果': [round(float(drift_score), 3), threshold, decision, active_model]
})

print(summary.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.hist(train_brightness, bins=bins, alpha=0.6, label='訓練資料亮度')
plt.hist(production_brightness, bins=bins, alpha=0.6, label='上線資料亮度')
plt.title('影像亮度分佈監控')
plt.xlabel('亮度')
plt.ylabel('數量')
plt.legend()
plt.show()


## 🧪 自我測驗

請完成下方 TODO 填空，依延遲、資料漂移與錯誤率三項監控指標，判斷生成式 AI 服務是否應該回退模型。


In [ ]:
# ── 自我測驗 ────────────────────────────────────
# 請完成下方 TODO 填空，實作簡化版的生成式 AI 服務風險判斷。

import numpy as np

# 情境：某生成式 AI 服務上線後，系統記錄了三個監控指標
avg_latency_ms = 950       # 平均延遲，單位毫秒
drift_score = 0.31         # 資料漂移分數
error_rate = 0.04          # 錯誤率

latency_limit = 800
drift_limit = 0.20
error_limit = 0.05

# TODO 1：判斷延遲是否超過門檻，將結果存入 latency_risk
latency_risk = ___

# TODO 2：判斷資料漂移是否超過門檻，將結果存入 drift_risk
drift_risk = ___

# TODO 3：只要延遲、資料漂移或錯誤率任一項超標，就應該回退模型
should_rollback = ___

if should_rollback:
    action = '回退到穩定版本'
else:
    action = '維持目前版本'

print('延遲風險:', latency_risk)
print('資料漂移風險:', drift_risk)
print('建議動作:', action)

# Expected:
# 延遲風險: True
# 資料漂移風險: True
# 建議動作: 回退到穩定版本
